# Market Selection Audit V2

This notebook documents the maintained market-selection workflow and moves the reusable block filters into a Python module under `polymarket_registry`.
The notebook itself is now only responsible for configuration, audit views, and block-by-block application.

## Paper Framing

For the NeurIPS version, the goal of filtering is not to remove every templated family of markets. The goal is to exclude market types that instantiate a world we do not want the model to treat as part of the core real-world belief system.

The most defensible exclusion factors are:

- **Ultra-short recurring template markets.** These markets are repeated mechanical contracts and contribute little meaningful real-world variation.
- **Financial price-derived or benchmark-derived markets.** These markets resolve as near-direct functions of ticker prices, benchmark levels, or structured price thresholds rather than broader world events.
- **Attention, speech, and mention-count markets.** These markets are often mediated by platform-specific measurement rules and noisy textual resolution criteria rather than substantive external state.
- **Sports and esports markets.** These markets are driven by highly idiosyncratic game-level dynamics and are only weakly coupled to the broader real-world state we want a foundation-style model to represent.

The final screen keeps only residual markets with volume above `20,000 USD`.

Import only the helpers we need. `polymarket_export` should already be installed in editable mode via `pip install -e ./polymarket_export`.

In [1]:
from pathlib import Path
import sqlite3

from IPython.display import display
import pandas as pd

import polymarket_registry

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

EXPORT_ROOT = Path(polymarket_registry.__file__).resolve().parents[1]
REPO_ROOT = EXPORT_ROOT.parent

EXPORT_ROOT, REPO_ROOT

(PosixPath('/Users/sneddy/research/polymarket_research/polymarket_export'),
 PosixPath('/Users/sneddy/research/polymarket_research'))

Configure the local database path and the main selection cutoffs.

In [2]:
DB_PATH = (REPO_ROOT / "db" / "resolved_probability_dataset.sqlite").resolve()
MIN_CREATED_AT = "2025-01-01T00:00:00Z"
MIN_RESIDUAL_VOLUME = 20_000.0

DB_PATH

PosixPath('/Users/sneddy/research/polymarket_research/db/resolved_probability_dataset.sqlite')

Load `market_universe` dynamically so the notebook stays compatible with schema changes.
We keep a preferred column order for readability, but automatically include any newly added columns.

In [3]:
preferred_column_order = [
    "market_id",
    "market_slug",
    "event_id",
    "event_slug",
    "event_title",
    "event_series_slug",
    "event_description",
    "event_score",
    "event_period",
    "event_series_id",
    "event_recurrence",
    "event_series_type",
    "question",
    "description",
    "volume_num",
    "outcomes",
    "outcome_prices",
    "uma_resolution_status",
    "neg_risk",
    "neg_risk_market_id",
    "group_item_title",
]

query = """
SELECT *
FROM market_universe
WHERE created_at IS NOT NULL
  AND created_at >= ?
ORDER BY volume_num DESC, created_at DESC
"""

with sqlite3.connect(DB_PATH) as conn:
    universe_columns = [row[1] for row in conn.execute("PRAGMA table_info(market_universe)").fetchall()]
    universe_df = pd.read_sql_query(query, conn, params=(MIN_CREATED_AT,))

extra_columns = [column for column in universe_df.columns if column not in preferred_column_order]
missing_preferred_columns = [column for column in preferred_column_order if column not in universe_df.columns]
ordered_columns = [column for column in preferred_column_order if column in universe_df.columns] + sorted(extra_columns)
universe_df = universe_df.loc[:, ordered_columns]

print({
    "db_path": str(DB_PATH),
    "universe_rows": len(universe_df),
    "min_created_at": MIN_CREATED_AT,
    "columns_in_table": len(universe_columns),
    "missing_preferred_columns": missing_preferred_columns,
    "extra_columns_detected": extra_columns,
})

universe_df.head(5)

{'db_path': '/Users/sneddy/research/polymarket_research/db/resolved_probability_dataset.sqlite', 'universe_rows': 732998, 'min_created_at': '2025-01-01T00:00:00Z', 'columns_in_table': 32, 'missing_preferred_columns': [], 'extra_columns_detected': ['condition_id', 'event_start_time', 'resolution_source', 'created_at', 'end_date', 'closed', 'archived', 'liquidity_num', 'clob_token_ids', 'closed_time', 'synced_at_utc']}


,market_id,market_slug,event_id,event_slug,event_title,event_series_slug,event_description,event_score,event_period,event_series_id,event_recurrence,event_series_type,question,description,volume_num,outcomes,outcome_prices,uma_resolution_status,neg_risk,neg_risk_market_id,group_item_title,archived,clob_token_ids,closed,closed_time,condition_id,created_at,end_date,event_start_time,liquidity_num,resolution_source,synced_at_utc
0,1640919,us-forces-enter-iran-by-april-30-899,158299,us-forces-enter-iran-by,US forces enter Iran by..?,NaN,"This market will resolve to “Yes” if active US military personnel physically enter Iran at any point by the listed date (ET). Otherwise, this market will re...",NaN,NaN,NaN,NaN,NaN,US forces enter Iran by April 30?,"This market will resolve to “Yes” if active US military personnel physically enter Iran at any point by the listed date (ET). Otherwise, this market will re...",2.690491e+08,"[""Yes"", ""No""]","[""1"", ""0""]",resolved,0.0,NaN,April 30,0,"[""2916184120206223749839849644877707470354946028257066951797428049170871002238"", ""76533108781962275310651165149634079251899733930834190485860627580128626747...",1,2026-04-09 00:28:21+00,0x6d0e09d0f04572d9b1adad84703458b0297bc5603b69dccbde93147ee4443246,2026-03-18T16:29:07.71272Z,2026-04-30T00:00:00Z,NaN,NaN,,2026-04-09T07:31:52Z
1,546814,will-zelenskyy-wear-a-suit-before-july,25044,will-zelenskyy-wear-a-suit-before-july,Will Zelenskyy wear a suit before July?,NaN,"This market will resolve to ""Yes"" if Volodymyr Zelenskyy is is photographed or videotaped wearing a suit between May 22 and June 30, 2025 ET. Otherwise, thi...",NaN,NaN,NaN,NaN,NaN,Will Zelenskyy wear a suit before July?,"This market will resolve to ""Yes"" if Volodymyr Zelenskyy is is photographed or videotaped wearing a suit between May 22 and June 30, 2025 ET. Otherwise, thi...",2.422312e+08,"[""Yes"", ""No""]","[""0"", ""1""]",resolved,0.0,NaN,,0,"[""103864131794756285503734468197278890131080300305704085735435172616220564121629"", ""343795817898955285602812182397592802372773053729787943248227774388244101...",1,2025-07-09 00:30:39+00,0x655e5ca101c466b6293aa15e06173b78b293221803d56e35551f708cd82eb352,2025-05-22T22:19:20.138837Z,2025-06-30T12:00:00Z,NaN,NaN,,2026-04-09T07:31:52Z
2,601697,fed-decreases-interest-rates-by-50-bps-after-january-2026-meeting,45883,fed-decision-in-january,Fed decision in January?,fed-interest-rates,The FED interest rates are defined in this market by the upper bound of the target federal funds range. The decisions on the target federal fund range are m...,NaN,NaN,35,monthly,single,Fed decreases interest rates by 50+ bps after January 2026 meeting?,The FED interest rates are defined in this market by the upper bound of the target federal funds range. The decisions on the target federal fund range are m...,2.350652e+08,"[""Yes"", ""No""]","[""0"", ""1""]",resolved,1.0,0x0d97e25f1830a3f18d5b4d43e6e1648d3b8b781e3c254a0b9d2c07a62f505100,50+ bps decrease,0,"[""11862165566757345985240476164489718219056735011698825377388402888080786399275"", ""7147885279027909544718299604907104079201075961766896979904917922910480057...",1,2026-01-28 22:53:04+00,0x17815081230e3b9c78b098162c33b1ffa68c4ec29c123d3d14989599e0c2e113,2025-09-17T16:22:35.837625Z,2026-01-28T00:00:00Z,NaN,NaN,,2026-04-09T07:31:52Z
3,601700,fed-increases-interest-rates-by-25-bps-after-january-2026-meeting,45883,fed-decision-in-january,Fed decision in January?,fed-interest-rates,The FED interest rates are defined in this market by the upper bound of the target federal funds range. The decisions on the target federal fund range are m...,NaN,NaN,35,monthly,single,Fed increases interest rates by 25+ bps after January 2026 meeting?,The FED interest rates are defined in this market by the upper bound of the target federal funds range. The decisions on the target federal fund range are m...,2.164557e+08,"[""Yes"", ""No""]","[""0"", ""1""]",resolved,1.0,0x0d97e25f1830a3f18d5b4d43e6e1648d3b8b781e3c254a0b9d2c07a62f505100

## Filter Imports

In [4]:
from polymarket_registry.block_filters import (
    AttentionSpeechMetricsFilter,
    FinalVolumeScreen,
    FinancialPriceDerivedFilter,
    SportsAndEsportsFilter,
    UltraShortRecurringTemplateFilter,
    WeatherMarketsFilter,
    initialize_work_df,
)

## Block 1. Ultra-Short Recurring Template Markets

In [5]:
UltraShortRecurringTemplateFilter.__doc__

'Exclude ultra-short recurring template markets.'

In [6]:
block1 = UltraShortRecurringTemplateFilter()
work_df = initialize_work_df(universe_df)

display(block1.show_recurrence_distribution(work_df))
display(block1.show_top_series_for_recurrence(work_df, "5m", 10))
display(block1.show_top_series_for_recurrence(work_df, "15m", 10))
display(block1.show_top_series_for_recurrence(work_df, "hourly", 10))

work_df = block1.apply_to_remaining(work_df)
block1.summarize_categories(work_df)

event_recurrence
daily      417700
5m         104686
15m         71151
NaN         66320
hourly      37888
weekly      27457
monthly      6596
annual       1200
Name: count, dtype: int64

event_series_slug
btc-up-or-down-5m    27641
sol-up-or-down-5m    25684
eth-up-or-down-5m    25681
xrp-up-or-down-5m    25680
Name: count, dtype: int64

event_series_slug
btc-up-or-down-15m    19929
eth-up-or-down-15m    19922
xrp-up-or-down-15m    15658
sol-up-or-down-15m    15642
Name: count, dtype: int64

event_series_slug
eth-up-or-down-hourly            7207
btc-up-or-down-hourly            7201
solana-up-or-down-hourly         7082
xrp-up-or-down-hourly            6898
bitcoin-multi-strikes-hourly     4569
ethereum-multi-strikes-hourly    4553
bitcoin-hourly-up-or-down         329
bitcoin-up-or-down-hourly          25
eth-hourly-up-or-down              24
Name: count, dtype: int64

category
unknown             477435
short_recurrence    255563
Name: count, dtype: int64

## Block 2. Financial Price-Derived Or Benchmark-Derived Markets

In [7]:
FinancialPriceDerivedFilter.__doc__

'Exclude price-derived and benchmark-derived financial markets.'

In [8]:
block2 = FinancialPriceDerivedFilter()

display(block2.show_flagged_series(work_df, 25))
display(block2.preview_flagged_rows(work_df, 10))

work_df = block2.apply_to_remaining(work_df)
block2.summarize_categories(work_df)

event_series_slug
ethereum-neg-risk-weekly         2285
ethereum-multi-strikes-weekly    2284
solana-multi-strikes-weekly      2259
solana-neg-risk-weekly           2249
xrp-multi-strikes-weekly         2244
bitcoin-neg-risk-weekly          2218
xrp-neg-risk-weekly              2210
bitcoin-multi-strikes-4h         1989
ethereum-multi-strikes-4h        1925
xrp-multi-strikes-4h             1925
solana-multi-strikes-4h          1925
btc-multi-strikes-weekly         1892
solana-neg-risk-4h               1342
bitcoin-neg-risk-4h              1331
ethereum-neg-risk-4h             1330
xrp-neg-risk-4h                  1309
bitcoin-hit-price-daily           582
ethereum-hit-price-weekly         529
bitcoin-hit-price-weekly          526
meta-multi-strikes-weekly         516
btc-daily-neg-risk                510
eth-daily-neg-risk                507
solana-hit-price-weekly           505
ethereum-hit-price-daily          434
xrp-hit-price-weekly              404
Name: count, dtype: int64

,market_id,market_slug,event_slug,event_series_slug,event_title,question,volume_num
80,1303355,will-bitcoin-reach-150k-in-february-2026,what-price-will-bitcoin-hit-in-february-2026,bitcoin-hit-price-monthly,What price will Bitcoin hit in February?,"Will Bitcoin reach $150,000 in February?",2.882922e+07
91,1082748,will-bitcoin-reach-150k-in-january-2026,what-price-will-bitcoin-hit-in-january-2026,bitcoin-hit-price-monthly,What price will Bitcoin hit in January?,"Will Bitcoin reach $150,000 in January?",2.666091e+07
102,1473040,will-bitcoin-reach-150k-in-march-2026,what-price-will-bitcoin-hit-in-march-2026,bitcoin-hit-price-monthly,What price will Bitcoin hit in March?,"Will Bitcoin reach $150,000 in March?",2.414935e+07
147,1467766,will-crude-oil-cl-hit-high-100-by-end-of-march-658-396-769-971,will-crude-oil-cl-hit-by-end-of-march,crude-oil-cl-hit,Will Crude Oil (CL) hit__ by end of March?,Will Crude Oil (CL) hit (HIGH) $100 by end of March?,1.659534e+07
156,618949,will-bitcoin-reach-200k-in-october,what-price-will-bitcoin-hit-in-october-985,btc-monthly-prices,What price will Bitcoin hit in October?,Will Bitcoin reach $200k in October?,1.561720e+07
184,659233,will-bitcoin-reach-200k-in-november-2025,what-price-will-bitcoin-hit-in-november-2025,bitcoin-hit-price-monthly,What price will Bitcoin hit in November?,"Will Bitcoin reach $200,000 in November?",1.358862e+07
193,1082749,will-ethereum-reach-6000-in-january-2026,what-price-will-ethereum-hit-in-january-2026,ethereum-hit-price-monthly,What price will Ethereum hit in January?,"Will Ethereum reach $6,000 in January?",1.321984e+07
197,1082781,will-bitcoin-reach-100k-in-january-2026,what-price-will-bitcoin-hit-in-january-2026,bitcoin-hit-price-monthly,What price will Bitcoin hit in January?,"Will Bitcoin reach $100,000 in January?",1.302559e+07
204,1516204,will-crude-oil-cl-hit-high-200-by-end-of-march-926,will-crude-oil-cl-hit-by-end-of-march,crude-oil-cl-hit,Will Crude Oil (CL) hit__ by end of March?,Will Crude Oil (CL) hit (HIGH) $200 by end of March?,1.248712e+07
227,631220,will-tesla-be-the-largest-company-in-the-world-by-market-cap-on-november-30,largest-company-end-of-november,largest-company,Largest Company end of November?,Will Tesla be the largest company in the world by market cap on November 30?,1.163768e+07


category
unknown                   429554
short_recurrence          255563
quant_price_structures     47881
Name: count, dtype: int64

## Block 3. Attention, Speech, And Mention-Count Markets

In [9]:
AttentionSpeechMetricsFilter.__doc__

'Exclude attention, speech, and mention-count markets.'

In [10]:
block3 = AttentionSpeechMetricsFilter()

display(block3.show_flagged_series(work_df, 25))
display(block3.preview_flagged_rows(work_df, 10))

work_df = block3.apply_to_remaining(work_df)
block3.summarize_categories(work_df)

event_series_slug
elon-tweets                2414
elon-tweets-48h             426
elon-tweet-daily            388
mrbeast-views-day-1         330
trump-weekly-mentions       330
all-in-podcast              263
starmer-pmqs                232
rogan-mentions              228
trump-truth-social          182
andrew-tate-tweets          176
trump-talk-monthly          145
mrbeast-views-week-1        126
trump-post-weekly           115
whitehouse-daily-tweets      62
trump-truths                 59
ted-cruz-daily-tweets        58
khamenei-daily-tweets        57
mrbeast-views                55
cz-tweets                    46
zelenskyy-tweets             45
nycmayor-tweets              44
potus-tweets                 27
powell-mentions              22
kanye-tweets                 12
Name: count, dtype: int64

,market_id,market_slug,event_slug,event_series_slug,event_title,question,volume_num
104,1423580,elon-musk-of-tweets-february-27-march-6-0-19,elon-musk-of-tweets-february-27-march-6,elon-tweets,"Elon Musk # tweets February 27 - March 6, 2026?","Will Elon Musk post 0-19 tweets from February 27 to March 6, 2026?",2.386434e+07
153,1405325,elon-musk-of-tweets-february-24-march-3-100-119,elon-musk-of-tweets-february-24-march-3,elon-tweets,"Elon Musk # tweets February 24 - March 3, 2026?","Will Elon Musk post 100-119 tweets from February 24 to March 3, 2026?",1.571330e+07
210,1455604,will-trump-talk-to-xi-jinping-in-march-165,who-will-trump-talk-to-in-march,trump-talk-monthly,Who will Trump talk to in March?,Will Trump talk to Xi Jinping in March?,1.235858e+07
393,1405322,elon-musk-of-tweets-february-24-march-3-80-99,elon-musk-of-tweets-february-24-march-3,elon-tweets,"Elon Musk # tweets February 24 - March 3, 2026?","Will Elon Musk post 80-99 tweets from February 24 to March 3, 2026?",7.682629e+06
407,920389,will-trump-talk-to-vladimir-putin-in-january,who-will-trump-talk-to-in-january,trump-talk-monthly,Who will Trump talk to in January?,Will Trump talk to Vladimir Putin in January?,7.456315e+06
444,900145,elon-musk-of-tweets-december-12-december-19-580plus,elon-musk-of-tweets-december-12-december-19,elon-tweets,"Elon Musk # tweets December 12 - December 19, 2025?","Will Elon Musk post 580+ tweets from December 12 to December 19, 2025?",7.049986e+06
630,1405319,elon-musk-of-tweets-february-24-march-3-60-79,elon-musk-of-tweets-february-24-march-3,elon-tweets,"Elon Musk # tweets February 24 - March 3, 2026?","Will Elon Musk post 60-79 tweets from February 24 to March 3, 2026?",5.276588e+06
671,1698788,elon-musk-of-tweets-march-27-april-3-20-39,elon-musk-of-tweets-march-27-april-3,elon-tweets,"Elon Musk # tweets March 27 - April 3, 2026?","Will Elon Musk post 20-39 tweets from March 27 to April 3, 2026?",5.041855e+06
677,924478,elon-musk-of-tweets-december-16-december-23-20-39,elon-musk-of-tweets-december-16-december-23,elon-tweets,"Elon Musk # tweets December 16 - December 23, 2025?","Will Elon Musk post 20-39 tweets from December 16 to December 23, 2025?",5.030123e+06
714,920396,will-trump-talk-to-emmanuel-macron-in-january,who-will-trump-talk-to-in-january,trump-talk-monthly,Who will Trump talk to in January?,Will Trump talk to Emmanuel Macron in January?,4.859960e+06


category
unknown                     423712
short_recurrence            255563
quant_price_structures       47881
attention_social_metrics      5842
Name: count, dtype: int64

## Block 4. Weather Markets

In [11]:
WeatherMarketsFilter.__doc__

'Exclude weather market families.'

In [12]:
block4 = WeatherMarketsFilter()

display(block4.show_flagged_series(work_df, 25))
display(block4.preview_flagged_rows(work_df, 10))

work_df = block4.apply_to_remaining(work_df)
block4.summarize_categories(work_df)

event_series_slug
nyc-daily-weather             3239
london-daily-weather          3226
toronto-daily-weather         1024
dallas-daily-weather          1021
seoul-daily-weather           1020
atlanta-daily-weather         1013
buenos-aires-daily-weather    1013
seattle-daily-weather         1009
miami-daily-weather            700
chicago-daily-weather          698
wellington-daily-weather       692
ankara-daily-weather           684
paris-daily-weather            524
sao-paulo-daily-weather        522
lucknow-daily-weather          352
munich-daily-weather           352
tokyo-daily-weather            313
tel-aviv-daily-weather         307
shanghai-daily-weather         287
singapore-daily-weather        285
hong-kong-daily-weather        260
taipei-daily-weather           257
milan-daily-weather            254
madrid-daily-weather           253
warsaw-daily-weather           253
Name: count, dtype: int64

,market_id,market_slug,event_slug,event_series_slug,event_title,question,volume_num
2453,528146,will-the-highest-temperature-in-london-be-53f-or-higher-on-march-17,highest-temperature-in-london-on-march-17,london-daily-weather,Highest temperature in London on March 17?,Will the highest temperature in London be 53°F or higher on March 17?,1.919499e+06
3691,1577678,highest-temperature-in-tel-aviv-on-march-16-2026-19c,highest-temperature-in-tel-aviv-on-march-16-2026,tel-aviv-daily-weather,Highest temperature in Tel Aviv on March 16?,Will the highest temperature in Tel Aviv be 19°C on March 16?,1.370313e+06
14199,553119,will-the-highest-temperature-in-london-be-77f-or-below-on-june-19,highest-temperature-in-london-on-june-19,london-daily-weather,Highest temperature in London on June 19?,Will the highest temperature in London be 77°F or below on June 19?,4.045044e+05
14357,522865,will-the-highest-temperature-in-nyc-be-between-35-36f-on-february-14,highest-temperature-in-nyc-on-feb-14,nyc-daily-weather,Highest temperature in NYC on Feb 14?,Will the highest temperature in NYC be between 35-36°F on February 14?,4.007192e+05
18524,1369027,highest-temperature-in-ankara-on-february-14-2026-7c,highest-temperature-in-ankara-on-february-14-2026,ankara-daily-weather,Highest temperature in Ankara on February 14?,Will the highest temperature in Ankara be 7°C on February 14?,3.060162e+05
20313,678305,will-the-highest-temperature-in-london-be-between-54-55f-on-november-14,highest-temperature-in-london-on-november-14,london-daily-weather,Highest temperature in London on November 14?,Will the highest temperature in London be between 54-55°F on November 14?,2.784292e+05
21491,1827150,highest-temperature-in-seoul-on-april-6-2026-13c,highest-temperature-in-seoul-on-april-6-2026,seoul-daily-weather,Highest temperature in Seoul on April 6?,Will the highest temperature in Seoul be 13°C on April 6?,2.625792e+05
21855,524127,will-the-highest-temperature-in-nyc-be-33f-or-below-on-february-22,highest-temperature-in-nyc-on-feb-22,nyc-daily-weather,Highest temperature in NYC on Feb 22?,Will the highest temperature in NYC be 33°F or below on February 22?,2.576614e+05
21911,834293,will-the-highest-temperature-in-london-be-between-54-55f-on-december-7,highest-temperature-in-london-on-december-7,london-daily-weather,Highest temperature in London on December 7?,Will the highest temperature in London be between 54-55°F on December 7?,2.570040e+05
21955,523310,will-the-highest-temperature-in-nyc-be-between-43-44f-on-february-16,highest-temperature-in-nyc-on-feb-16,nyc-daily-weather,Highest temperature in NYC on Feb 16?,Will the highest temperature in NYC be between 43-44°F on February 16?,2.564622e+05


category
unknown                     401492
short_recurrence            255563
quant_price_structures       47881
weather                      22220
attention_social_metrics      5842
Name: count, dtype: int64

## Block 5. Sports And Esports Markets

In [13]:
SportsAndEsportsFilter.__doc__

'Exclude sports and esports markets.'

In [14]:
block5 = SportsAndEsportsFilter()

display(block5.show_flagged_split(work_df))
display(block5.preview_flagged_rows(work_df, 10))

work_df = block5.apply_to_remaining(work_df)
block5.summarize_categories(work_df)

cybersport     97482
sport         239302
dtype: int64

,market_id,market_slug,event_slug,event_series_slug,event_title,question,volume_num
16,566189,will-chelsea-win-the-202526-english-premier-league,english-premier-league-winner,NaN,English Premier League Winner,Will Chelsea win the 2025–26 English Premier League?,9.127917e+07
63,566203,will-leeds-win-the-202526-english-premier-league,english-premier-league-winner,NaN,English Premier League Winner,Will Leeds win the 2025–26 English Premier League?,3.835935e+07
67,1269423,nfl-sea-ne-2026-02-08,nfl-sea-ne-2026-02-08,nfl-2025,Seattle vs. New England,Seahawks vs. Patriots,3.628175e+07
73,566192,will-tottenham-win-the-202526-english-premier-league,english-premier-league-winner,NaN,English Premier League Winner,Will Tottenham win the 2025–26 English Premier League?,3.034374e+07
84,660880,lol-t1-kt-2025-11-09,lol-t1-kt-2025-11-09,league-of-legends,LoL: T1 vs KT Rolster (BO5),LoL: T1 vs KT Rolster (BO5),2.823290e+07
95,553878,will-the-chicago-bulls-win-the-2026-nba-finals,2026-nba-champion,NaN,2026 NBA Champion,Will the Chicago Bulls win the 2026 NBA Finals?,2.535091e+07
129,566242,will-mallorca-win-the-202526-la-liga,la-liga-winner-114,NaN,LA LIGA Winner,Will Mallorca win the 2025–26 La Liga?,1.865490e+07
136,520358,nfl-kc-phi-2025-02-09,nfl-kc-phi-2025-02-09,nfl,Super Bowl LIX Winner,Super Bowl LIX Winner,1.753857e+07
149,566194,will-brighton-win-the-202526-english-premier-league,english-premier-league-winner,NaN,English Premier League Winner,Will Brighton win the 2025–26 English Premier League?,1.647172e+07
151,566198,will-west-ham-win-the-202526-english-premier-league,english-premier-league-winner,NaN,English Premier League Winner,Will West Ham win the 2025–26 English Premier League?,1.586056e+07


category
short_recurrence            255563
sport                       239302
cybersport                   97482
unknown                      64708
quant_price_structures       47881
weather                      22220
attention_social_metrics      5842
Name: count, dtype: int64

## Final Volume Screen (> 20,000 USD)

In [15]:
FinalVolumeScreen.__doc__

'Exclude low-volume residual markets at the end of the pipeline.'

In [16]:
volume_filter = FinalVolumeScreen(MIN_RESIDUAL_VOLUME)

display(volume_filter.show_low_volume_examples(work_df, 20))

work_df = volume_filter.apply_to_remaining(work_df)
volume_filter.summarize_categories(work_df)

,market_id,event_series_slug,event_title,question,volume_num
139948,580524,israel-strike-yemen,Israel strikes Yemen again by September 15?,Israel strikes Yemen again by September 15?,19999.623311
139949,700302,NaN,Qatar Grand Prix: Sprint Winner,Will Jack Doohan win the Sprint at the 2025 F1 Qatar Grand Prix?,19999.602016
139959,566040,NaN,Which Golfers will make USA Ryder Cup team?,Will Harris English make the 2025 USA Ryder Cup team?,19996.000000
139966,547080,trump-538-approval,Trump approval rating on May 30?,Will Trump's approval rating be between 45.5% and 45.9% on May 30?,19994.010442
139967,527262,jobs-added,How many jobs added in March?,Will the US add less than 50k jobs in March?,19993.735550
139979,971321,israel-strike-lebanon-on,Will Israel strike Lebanon on...?,Will Israel strike Lebanon on December 28?,19988.522108
139986,915686,NaN,Who will win the Aster trading competition?,Will 0xJack win the Aster trading competition?,19985.819092
140013,540021,NaN,Will Canada raise tariffs on the U.S. before June?,Will Canada raise tariffs on the U.S. before June?,19978.212913
140026,1249335,trump-538-approval,Trump approval rating on January 30?,"Will Trump's approval rating be between 41.5 and 41.9 on January 30, 2026?",19973.848514
140032,991041,central-bank-of-colombia-decision,Central Bank of Colombia decision in January?,Will the Central Bank of Colombia announce a decrease at the January meeting?,19971.878823


category
short_recurrence            255563
sport                       239302
cybersport                   97482
quant_price_structures       47881
low_volume                   45933
weather                      22220
unknown                      18775
attention_social_metrics      5842
Name: count, dtype: int64

## Residual Review After All Filters

In [17]:
rest = work_df[work_df["category"].isna()].copy()

print({
    "final_residual_rows": len(rest),
    "min_volume_usd": MIN_RESIDUAL_VOLUME,
})

display(work_df["category"].fillna("kept").value_counts())
display(rest["event_series_slug"].fillna("<NA>").value_counts().head(100))

if len(rest) > 0:
    display(rest.sample(min(len(rest), 5)))
    display(rest[rest["event_series_slug"].isna()].head(10))

{'final_residual_rows': 18775, 'min_volume_usd': 20000.0}


category
short_recurrence            255563
sport                       239302
cybersport                   97482
quant_price_structures       47881
low_volume                   45933
weather                      22220
kept                         18775
attention_social_metrics      5842
Name: count, dtype: int64

event_series_slug
<NA>                                                         13736
box-office-openings                                            351
uef-qualifiers                                                 146
next-country-us-strike                                         120
israel-strike-lebanon-on                                       120
israel-strike-gaza-on                                          112
trump-538-approval                                             109
us-strikes-iran                                                 99
ufc                                                             79
trump-daily-eos                                                 76
trump-monthly-meeting                                           74
second-best-ai-company                                          69
top-ai-company                                                  67
top-ai-company-style-on                                         66
named-in-newly-released-epstein-files       

,market_id,market_slug,event_id,event_slug,event_title,event_series_slug,event_description,event_score,event_period,event_series_id,event_recurrence,event_series_type,question,description,volume_num,outcomes,outcome_prices,uma_resolution_status,neg_risk,neg_risk_market_id,group_item_title,archived,clob_token_ids,closed,closed_time,condition_id,created_at,end_date,event_start_time,liquidity_num,resolution_source,synced_at_utc,category
70289,1059090,lighter-airdrop-on-january-10-559,131448,what-day-will-the-lighter-airdrop-be-489,What day will the Lighter airdrop be? (2026),NaN,"This market will resolve according to the next date, in ET, on which Lighter performs an airdrop on. \n\nIf Lighter launches a memecoin and performs an aird...",NaN,NaN,NaN,NaN,NaN,Lighter Airdrop on January 10?,"This market will resolve according to the next date, in ET, on which Lighter performs an airdrop on. \n\nIf Lighter launches a memecoin and performs an aird...",72232.337700,"[""Yes"", ""No""]","[""0"", ""1""]",resolved,1.0,0x8bb1ca1e17bb411d09e11f596036891e1457726d40f6eb8f63eb1b8f0b2c2a00,January 10,0,"[""111757207475925659031898273265400541528076474912853693741876962719003506809568"", ""271020219053930448673090219622216112783245665279920658275209638573352210...",1,2025-12-30 10:45:29+00,0xb09b2a3ec1d0b31b9954c833a1def3f9d054b174fe24f24bea2236c808452e89,2025-12-29T18:16:34.817763Z,2027-01-01T05:00:00Z,NaN,NaN,NaN,2026-04-09T07:31:52Z,None
6907,525013,will-mark-ouellet-be-the-next-pope,19581,who-will-be-the-next-pope,Who will be the next Pope?,NaN,"This market will resolve to the name of the next person elected to be the bishop of Rome after Pope Francis.\n\nIf no new pope is elected by December 31, 20...",NaN,NaN,NaN,NaN,NaN,Will Mark Ouellet be the next pope?,"This market will resolve to the name of the next person announced as the bishop of Rome after Pope Francis.\n\nIf no new pope is announced by December 31, 2...",794266.416199,"[""Yes"", ""No""]","[""0"", ""1""]",resolved,1.0,0x224c5e7018556c551234689745e85e4844ef4fb5778ff5c2cbe2a689d7ac0900,Mark Ouellet,0,"[""2662352934415897199917947053028270362807640896247067452989102102054410542381"", ""48298055821090584965865785350691154723675981204672577745853309071408918993...",1,2025-05-08 22:41:29+00,0x843f8eb5959f17324cd6272b3665aa9c377e28d8d2d94ec120aceb745f3fdb9c,2025-02-24T17:19:01.437378Z,2025-12-31T12:00:00Z,2025-05-07T06:00:00Z,NaN,,2026-04-09T07:31:52Z,None
33122,550464,ufc-fight-night-namajunas-vs-maverick,26268,ufc-fight-night-usman-vs-buckley,UFC Fight Night: Usman vs. Buckley,NaN,This is a market on the outcomes and results of the UFC Fight Night: Usman vs. Buckley event.,NaN,NaN,NaN,NaN,NaN,UFC Fight Night: Namajunas vs. Maverick,"This is a market on whether Rose Namajunas vs. Miranda Maverick will win their bout at UFC Fight Night: Usman vs. Buckley, scheduled for June 14, 2025. \n\n...",169283.800954,"[""Namajunas"", ""Maverick""]","[""1"", ""0""]",resolved,0.0,NaN,Namajunas vs. Maverick,0,"[""35204238028912546811419904449682972970007061332823060653105857410320605809878"", ""5405683193572679356469414910308785605226321830386566391719905381419147539...",1,2025-06-15 06:39:50+00,0x58a845ca477ab4348ce4ec351d49ca36ebd2dc7b15223c9bc23a492d5644e2db,2025-06-07T02:42:23.483992Z,2025-06-15T00:00:00Z,NaN,NaN,,2026-04-09T07:31:52Z,None
113071,647870,will-a-candidate-win-outright-in-irelands-first-round,64572,will-a-candidate-win-outright-in-irelands-first-round,Will a candidate win outright in Ireland’s first round?,NaN,"The 2025 Irish presidential election is scheduled for October 24, 2025. The President of Ireland is elected via an instant-runoff voting (single transferabl...",NaN,NaN,NaN,NaN,NaN,Will a candidate win outright in Ireland’s first round?,"The 2025 Irish presidential election is scheduled for October 24, 2025. The President of Ireland is elected via an instant-runoff voting (single transferabl...",31159.683031,"[""Yes"", ""No""]","[""1"", ""0""]",resolved,0.0,NaN,,0,"[""29380422

,market_id,market_slug,event_id,event_slug,event_title,event_series_slug,event_description,event_score,event_period,event_series_id,event_recurrence,event_series_type,question,description,volume_num,outcomes,outcome_prices,uma_resolution_status,neg_risk,neg_risk_market_id,group_item_title,archived,clob_token_ids,closed,closed_time,condition_id,created_at,end_date,event_start_time,liquidity_num,resolution_source,synced_at_utc,category
0,1640919,us-forces-enter-iran-by-april-30-899,158299,us-forces-enter-iran-by,US forces enter Iran by..?,NaN,"This market will resolve to “Yes” if active US military personnel physically enter Iran at any point by the listed date (ET). Otherwise, this market will re...",NaN,NaN,NaN,NaN,NaN,US forces enter Iran by April 30?,"This market will resolve to “Yes” if active US military personnel physically enter Iran at any point by the listed date (ET). Otherwise, this market will re...",2.690491e+08,"[""Yes"", ""No""]","[""1"", ""0""]",resolved,0.0,NaN,April 30,0,"[""2916184120206223749839849644877707470354946028257066951797428049170871002238"", ""76533108781962275310651165149634079251899733930834190485860627580128626747...",1,2026-04-09 00:28:21+00,0x6d0e09d0f04572d9b1adad84703458b0297bc5603b69dccbde93147ee4443246,2026-03-18T16:29:07.71272Z,2026-04-30T00:00:00Z,NaN,NaN,,2026-04-09T07:31:52Z,None
1,546814,will-zelenskyy-wear-a-suit-before-july,25044,will-zelenskyy-wear-a-suit-before-july,Will Zelenskyy wear a suit before July?,NaN,"This market will resolve to ""Yes"" if Volodymyr Zelenskyy is is photographed or videotaped wearing a suit between May 22 and June 30, 2025 ET. Otherwise, thi...",NaN,NaN,NaN,NaN,NaN,Will Zelenskyy wear a suit before July?,"This market will resolve to ""Yes"" if Volodymyr Zelenskyy is is photographed or videotaped wearing a suit between May 22 and June 30, 2025 ET. Otherwise, thi...",2.422312e+08,"[""Yes"", ""No""]","[""0"", ""1""]",resolved,0.0,NaN,,0,"[""103864131794756285503734468197278890131080300305704085735435172616220564121629"", ""343795817898955285602812182397592802372773053729787943248227774388244101...",1,2025-07-09 00:30:39+00,0x655e5ca101c466b6293aa15e06173b78b293221803d56e35551f708cd82eb352,2025-05-22T22:19:20.138837Z,2025-06-30T12:00:00Z,NaN,NaN,,2026-04-09T07:31:52Z,None
7,538932,will-zohran-mamdani-win-the-2025-nyc-mayoral-election,23246,new-york-city-mayoral-election,New York City Mayoral Election,NaN,"The 2025 New York City mayoral election will be held on November 4, 2025.\n\nThis market will resolve according to the candidate wins the election.\n\nThe p...",NaN,NaN,NaN,NaN,NaN,Will Zohran Mamdani win the 2025 NYC mayoral election?,"The 2025 New York City mayoral election will be held on November 4, 2025, to elect the mayor of New York City.\n\nThis market will resolve according to the ...",1.432549e+08,"[""Yes"", ""No""]","[""1"", ""0""]",resolved,1.0,0x289dc66d9938b32cb340a630d13de0a91e3e04e0dc62a219a26a16e6f9cc2600,Zohran Mamdani,0,"[""33945469250963963541781051637999677727672635213493648594066577298999471399137"", ""1058323623507886161486123626429924039967140209185589172751517461775255187...",1,2025-11-05 05:44:47+00,0xebddfcf7b4401dade8b4031770a1ab942b01854f3bed453d5df9425cd9f211a9,2025-04-22T15:32:27.448351Z,2025-11-04T12:00:00Z,2025-11-04T12:00:00Z,NaN,,2026-04-09T07:31:52Z,None
10,572473,will-trump-nominate-judy-shelton-as-the-next-fed-chair,35908,who-will-trump-nominate-as-fed-chair,Who will Trump nominate as Fed Chair?,NaN,"This market will resolve according to the next individual Donald Trump, as President of the United States, formally nominates to be Chair of the Federal Res...",NaN,NaN,NaN,NaN,NaN,Will Trump nominate Judy Shelton as the next Fed chair?,"This market will resolve according to the next individual Donald Trump, as President of the United States, formally nominates to be Chair of the Federal Res...",1.276841e+08,"[""Yes"", ""No""]","[""0"", ""1""]",resolved,1.0,0x4714f4189125bba4cb9e6f9e8b5757ebd34a5be31379c33a665e4b0ca9738600,Ju

In [18]:
rest['outcome_prices']

0         ["1", "0"]
1         ["0", "1"]
2         ["0", "1"]
3         ["0", "1"]
4         ["0", "1"]
             ...    
139878    ["0", "1"]
139882    ["0", "1"]
139894    ["0", "1"]
139908    ["0", "1"]
139909    ["0", "1"]
Name: outcome_prices, Length: 18775, dtype: str

In [20]:
import ast
import numpy as np
import pandas as pd

def parse_list(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        try:
            return ast.literal_eval(text)
        except Exception:
            return None
    return None

def parse_binary_prices(value):
    parsed = parse_list(value)
    if not isinstance(parsed, list) or len(parsed) != 2:
        return None
    try:
        out = [float(x) for x in parsed]
    except Exception:
        return None
    if not all(np.isfinite(x) for x in out):
        return None
    return out

def parse_binary_outcomes(value):
    parsed = parse_list(value)
    if not isinstance(parsed, list) or len(parsed) != 2:
        return None
    out = [str(x).strip() for x in parsed]
    return out if {x.lower() for x in out} == {"yes", "no"} else None

def resolved_outcome_from_prices(prices, outcomes, threshold=0.99):
    if len(prices) != 2 or len(outcomes) != 2:
        return None, None
    if abs(sum(prices) - 1.0) > 1e-3:
        return None, None
    winner_idx = int(np.argmax(prices))
    winner_prob = float(prices[winner_idx])
    if winner_prob < threshold:
        return None, None
    return outcomes[winner_idx], winner_prob


In [21]:

rest = work_df.loc[work_df["category"].isna()].copy()

rest["parsed_prices"] = rest["outcome_prices"].map(parse_binary_prices)
rest["parsed_outcomes"] = rest["outcomes"].map(parse_binary_outcomes)

pairs = [
    resolved_outcome_from_prices(p, o)
    if p is not None and o is not None
    else (None, None)
    for p, o in zip(rest["parsed_prices"], rest["parsed_outcomes"], strict=False)
]

rest["final_outcome"] = [x[0] for x in pairs]
rest["winner_prob"] = [x[1] for x in pairs]
rest["final_yes_probability"] = [
    float(p[o.index("Yes")]) if (p is not None and o is not None and x[0] is not None) else None
    for p, o, x in zip(rest["parsed_prices"], rest["parsed_outcomes"], pairs, strict=False)
]


In [22]:
rest["has_final_outcome"] = rest["final_outcome"].notna()

rest["has_final_outcome"].value_counts(dropna=False)


has_final_outcome
True     17247
False     1528
Name: count, dtype: int64

In [24]:
rest[rest.has_final_outcome == False].sample(10)

,market_id,market_slug,event_id,event_slug,event_title,event_series_slug,event_description,event_score,event_period,event_series_id,event_recurrence,event_series_type,question,description,volume_num,outcomes,outcome_prices,uma_resolution_status,neg_risk,neg_risk_market_id,group_item_title,archived,clob_token_ids,closed,closed_time,condition_id,created_at,end_date,event_start_time,liquidity_num,resolution_source,synced_at_utc,category,parsed_prices,parsed_outcomes,final_outcome,winner_prob,final_yes_probability,has_final_outcome
55825,526687,brunno-ferreira-vs-armen-petrosyan,18636,ufc-313-pereria-vs-ankalaev,UFC 313: Pereira vs. Ankalaev,NaN,This is a market on the outcome of the UFC 313 fight between Pereira vs. Ankalaev.,NaN,NaN,NaN,NaN,NaN,Brunno Ferreira vs. Armen Petrosyan,"This is a market on whether Brunno Ferreira or Armen Petrosyan will win their bout at UFC 313: Pereira vs. Ankalaev, scheduled for March 8, 2025, at T-Mobil...",98064.613299,"[""Ferreira"", ""Petrosyan""]","[""1"", ""0""]",resolved,0.0,NaN,Ferreira vs. Petrosyan,0,"[""31338082215762382351756296155303816111324285573024827694400151864584880590255"", ""1076792781902602378604590508482745904034786079984018834525168650346350114...",1,2025-03-09 04:45:04+00,0x36e88e304ac6f9019bbb5570d070810af217cc56aebe3943fb2383b7b0ca1c7a,2025-03-03T22:49:38.292702Z,2025-03-08T12:00:00Z,NaN,NaN,,2026-04-09T07:31:52Z,None,"[1.0, 0.0]",None,NaN,NaN,NaN,False
116990,560689,will-bitcoin-hit-108k-or-109k-first,31342,will-bitcoin-hit-108k-or-109k-first,Will Bitcoin hit $108k or $109k first?,NaN,"This is a market on whether Bitcoin ($BTC) will first hit $108,000.00 or $109,000.00 first between July 7, 2025, 08:00 AM ET, and July 31, 2025, 11:59 PM ET...",NaN,NaN,NaN,NaN,NaN,Will Bitcoin hit $108k or $109k first?,"This is a market on whether Bitcoin ($BTC) will first hit $108,000.00 or $109,000.00 first between July 7, 2025, 08:00 AM ET, and July 31, 2025, 11:59 PM ET...",29047.327566,"[""108k"", ""109k""]","[""1"", ""0""]",resolved,0.0,NaN,,0,"[""95117275442286875105258926349700971301020742223781261229802216118618555716092"", ""3766838010941717551078404868160346374575128499711230942059088475941453409...",1,2025-07-07 16:43:48+00,0x8fb9f6085e34d50c0ecd09ddaaa3a7e947c3941c7d8d3d3913b21b4a07483a25,2025-07-07T12:55:59.05288Z,2025-07-31T00:00:00Z,NaN,NaN,,2026-04-09T07:31:52Z,None,"[1.0, 0.0]",None,NaN,NaN,NaN,False
112248,529849,gujarat-titans-vs-mumbai-indians,21398,gujarat-titans-vs-mumbai-indians,Gujarat Titans vs Mumbai Indians,NaN,"In the upcoming IPL game, scheduled for March 29 at 10:00 AM ET:\n\nIf the Gujarat Titans win, the market will resolve to “Gujarat”.\n\nIf the Mumbai Indian...",NaN,NaN,NaN,NaN,NaN,Gujarat Titans vs Mumbai Indians,"In the upcoming IPL game, scheduled for March 29 at 10:00 AM ET:\n\nIf the Gujarat Titans win, the market will resolve to “Gujarat”.\n\nIf the Mumbai Indian...",31635.425135,"[""Gujarat"", ""Mumbai""]","[""1"", ""0""]",resolved,0.0,NaN,,0,"[""20327196650534943673708075281861205738603368410661298652407536838522741193111"", ""6914731125440926858858647152499288741103413009283226771563420845749961254...",1,2025-03-29 21:01:49+00,0x680cd43bdf64ac026e793c130eacf7a1d91d5ee25f771ea12d96b0897182c2d3,2025-03-22T22:12:07.736031Z,2025-03-29T12:00:00Z,NaN,NaN,https://www.iplt20.com/,2026-04-09T07:31:52Z,None,"[1.0, 0.0]",None,NaN,NaN,NaN,False
124542,658171,speed-chess-qf-nakamura-vs-so,69714,speed-chess-qf-nakamura-vs-so,Speed Chess QF: Nakamura vs So,speed-chess-qf,"The 2025–2026 Speed Chess Championship is currently scheduled to take place from October 13, 2025, to February 9, 2026, at 11:59 PM ET.\n\nThis market will ...",NaN,NaN,10491,daily,single,Speed Chess QF: Nakamura vs So,"The 2025–2026 Speed Chess Championship is currently scheduled to take place from October 13, 2025, to February 9, 2026, at 11:59 PM ET.\n\nThis market will ...",25491.103972,"[""Nakamura"", ""So""]","[""1"", ""0""]",resolved,0.0,NaN,,0,"[""672906449592955953084834652845914518

In [25]:
rest["outcome_type"] = rest["parsed_outcomes"].map(
    lambda x: "binary_yes_no" if x is not None else "other_or_unparsed"
)
rest["outcome_type"].value_counts()


outcome_type
binary_yes_no        17255
other_or_unparsed     1520
Name: count, dtype: int64

In [27]:
rest.loc[
    rest["parsed_outcomes"].notna() & rest["final_outcome"].isna(),
    ["market_id", "market_slug", "question", "outcomes", "outcome_prices"]
]

,market_id,market_slug,question,outcomes,outcome_prices
1652,701458,megaeth-250m-pre-deposit-bridge-filled-in-15min,MegaETH $250M pre-deposit bridge filled in 15 minutes?,"[""Yes"", ""No""]","[""0.5"", ""0.5""]"
18949,524476,will-bybit-buy-1b-of-eth,Will Bybit buy >$1b ETH by next Friday?,"[""Yes"", ""No""]","[""0.5"", ""0.5""]"
59086,541114,will-the-next-pope-be-in-favor-of-making-priestly-celibacy-optional,Will the next Pope be in favor of Making Priestly Celibacy Optional?,"[""Yes"", ""No""]","[""0.5"", ""0.5""]"
59188,541120,will-the-next-pope-be-in-favor-of-communion-for-divorced-remarried,Will the next Pope be in favor of Communion for Divorced & Remarried?,"[""Yes"", ""No""]","[""0.5"", ""0.5""]"
79354,541119,will-the-next-pope-be-in-favor-of-reassessing-humanae-vitae,Will the next Pope be in favor of Reassessing Humanae Vitae?,"[""Yes"", ""No""]","[""0.5"", ""0.5""]"
81102,541121,will-the-next-pope-be-in-favor-of-the-german-synodal-way,Will the next Pope be in favor of the German “Synodal Way”?,"[""Yes"", ""No""]","[""0.5"", ""0.5""]"
114518,674712,ufc-cod11-mal3-2025-11-15-wellmaker-win-by-ko-tko,Will Malcolm Wellmaker win by KO or TKO?,"[""Yes"", ""No""]","[""0.5"", ""0.5""]"
126838,1155827,ufc-dus3-jim-2026-01-31-crute-win-by-ko-tko,Will Jimmy Crute win by KO or TKO?,"[""Yes"", ""No""]","[""0.5"", ""0.5""]"


In [30]:
rest['outcome_prices'].value_counts()

outcome_prices
["0", "1"]        13403
["1", "0"]         5310
["0.5", "0.5"]       62
Name: count, dtype: int64

In [32]:
rest['outcomes'].value_counts().head(50)

outcomes
["Yes", "No"]                                              17255
["Trzaskowski", "Nawrocki"]                                    4
["Pakistan", "South Africa"]                                   4
["Sabalenka", "Anisimova"]                                     3
["Finland", "Switzerland"]                                     3
["No Change", "25bps cut"]                                     3
["The MongolZ", "Liquid"]                                      3
["The MongolZ", "Aurora"]                                      3
["USA", "Norway"]                                              3
["G2", "FURIA"]                                                3
["Nothing", "Something"]                                       3
["Boy", "Girl"]                                                3
["FaZe", "Liquid"]                                             3
["Australia", "India"]                                         3
["Canada", "USA"]                                              2
["Canada", "Czec